# 📈 ใบงานสัปดาห์ที่ 08 — นาอีฟเบย์ส (Naive Bayes Classifier)

---

**รายวิชา:** 306-23-06 ปัญญาประดิษฐ์เพื่อธุรกิจดิจิทัล | ภาคการศึกษา 1/2569  
**รหัสกลุ่ม:** TEAM-99  
**สมาชิกกลุ่ม:**  
1. น.ส.กมลวรรณ จันทร์ผึ้ง (001)  
2. น.ส.ชลดา อิศรเสนา ณ อยุธยา (009)  
3. น.ส.ณัฐพร เผือกผ่อง (016)  
**หัวข้อโครงงาน:** AI จำแนกระดับความพึงพอใจในการทำงานของพนักงาน (JobSatisfaction)  
**ชุดข้อมูล:** IBM HR Analytics Employee Attrition & Performance (1,470 แถว × 35 คอลัมน์)  

---

### 📌 วัตถุประสงค์และภาพรวม
การจำแนกระดับความพึงพอใจด้วย GaussianNB วิเคราะห์ Prior Probabilities และตารางค่าเฉลี่ยฟีเจอร์ theta_


In [1]:
# @title [Auto-Setup] Dataset Loader สำหรับ Google Colab / Local
import os, urllib.request, pandas as pd, shutil

DATA_URL = 'https://raw.githubusercontent.com/ibm-watson-data-lab/open-data/master/HR-Employee-Attrition.csv'

# 1. เตรียม Raw Dataset
os.makedirs('data', exist_ok=True)
if not os.path.exists('WA_Fn-UseC_-HR-Employee-Attrition.csv') and not os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv'):
    try:
        print('Downloading raw dataset...')
        urllib.request.urlretrieve(DATA_URL, 'WA_Fn-UseC_-HR-Employee-Attrition.csv')
    except Exception as e:
        print('Note:', e)

if os.path.exists('WA_Fn-UseC_-HR-Employee-Attrition.csv') and not os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv'):
    shutil.copyfile('WA_Fn-UseC_-HR-Employee-Attrition.csv', 'data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
elif os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv') and not os.path.exists('WA_Fn-UseC_-HR-Employee-Attrition.csv'):
    shutil.copyfile('data/WA_Fn-UseC_-HR-Employee-Attrition.csv', 'WA_Fn-UseC_-HR-Employee-Attrition.csv')

# 2. เตรียม clean.csv / clean (4).csv สำหรับโมเดล
if not os.path.exists('clean.csv') and not os.path.exists('clean (4).csv'):
    raw_path = 'data/WA_Fn-UseC_-HR-Employee-Attrition.csv' if os.path.exists('data/WA_Fn-UseC_-HR-Employee-Attrition.csv') else 'WA_Fn-UseC_-HR-Employee-Attrition.csv'
    if os.path.exists(raw_path):
        raw_df = pd.read_csv(raw_path)
        cols_to_drop = [c for c in ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber'] if c in raw_df.columns]
        clean_df = raw_df.drop(columns=cols_to_drop)
        num_cols = clean_df.select_dtypes(include=['int64', 'float64']).columns
        cat_cols = clean_df.select_dtypes(include=['object']).columns
        clean_df = pd.get_dummies(clean_df, columns=cat_cols, drop_first=True)
        clean_df.to_csv('clean.csv', index=False)

if os.path.exists('clean.csv'):
    if not os.path.exists('clean (4).csv'): shutil.copyfile('clean.csv', 'clean (4).csv')
    if not os.path.exists('clean (3).csv'): shutil.copyfile('clean.csv', 'clean (3).csv')

print(' Dataset ready for execution!')


In [2]:
import pandas as pd
df = pd.read_csv("clean (4).csv")
features = [
    "Age","MonthlyIncome","DistanceFromHome", "TotalWorkingYears","YearsAtCompany"]
X = df[features]
y = df["JobSatisfaction"].astype(int)
print(  df.shape," จำนวนแต่ละระดับความพึงพอใจ:")

print(y.value_counts().sort_index())
print("สัดส่วนแต่ละระดับ:")
print(   y.value_counts(normalize=True).sort_index().round(3))


(1470, 43)  จำนวนแต่ละระดับความพึงพอใจ:
JobSatisfaction
1    289
2    280
3    442
4    459
Name: count, dtype: int64
สัดส่วนแต่ละระดับ:
JobSatisfaction
1    0.197
2    0.190
3    0.301
4    0.312
Name: proportion, dtype: float64


In [3]:
print("ค่าหายในฟีเจอร์:", X.isna().sum().sum())
X.dtypes

ค่าหายในฟีเจอร์: 0


,0
Age,float64
MonthlyIncome,int64
DistanceFromHome,int64
TotalWorkingYears,int64
YearsAtCompany,int64


In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

(1176, 5) (294, 5)


In [5]:
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()
model.fit(X_train, y_train)

GaussianNB()

In [6]:
from sklearn.metrics import accuracy_score
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
print("Naive Bayes test accuracy:",round(acc, 3))

Naive Bayes test accuracy: 0.296


In [7]:
from sklearn.metrics import confusion_matrix, classification_report
print(  confusion_matrix(y_test, pred,labels=[1, 2, 3, 4]))
print( classification_report( y_test, pred,labels=[1, 2, 3, 4], target_names=[
            "ระดับ 1",
            "ระดับ 2",
            "ระดับ 3",
            "ระดับ 4 "  ],  zero_division=0))

[[ 0  2 16 40]
 [ 0  2 11 43]
 [ 0  0 27 61]
 [ 0  1 33 58]]
              precision    recall  f1-score   support

     ระดับ 1       0.00      0.00      0.00        58
     ระดับ 2       0.40      0.04      0.07        56
     ระดับ 3       0.31      0.31      0.31        88
    ระดับ 4        0.29      0.63      0.39        92

    accuracy                           0.30       294
   macro avg       0.25      0.24      0.19       294
weighted avg       0.26      0.30      0.23       294



In [8]:
proba = model.predict_proba(X_test[:5])
for i, p in enumerate(proba):
    print(
        "P[ระดับ 1, ระดับ 2, ระดับ 3, ระดับ 4] =",
        p.round(3), "-> ทำนาย:",
        pred[i],"จริง:",y_test.iloc[i] )

P[ระดับ 1, ระดับ 2, ระดับ 3, ระดับ 4] = [0.224 0.171 0.279 0.326] -> ทำนาย: 4 จริง: 3
P[ระดับ 1, ระดับ 2, ระดับ 3, ระดับ 4] = [0.187 0.188 0.296 0.328] -> ทำนาย: 4 จริง: 4
P[ระดับ 1, ระดับ 2, ระดับ 3, ระดับ 4] = [0.21  0.173 0.292 0.325] -> ทำนาย: 4 จริง: 1
P[ระดับ 1, ระดับ 2, ระดับ 3, ระดับ 4] = [0.206 0.179 0.303 0.312] -> ทำนาย: 4 จริง: 1
P[ระดับ 1, ระดับ 2, ระดับ 3, ระดับ 4] = [0.204 0.175 0.294 0.328] -> ทำนาย: 4 จริง: 4


In [9]:
print("คลาส:", model.classes_)
print(  "prior (สัดส่วนก่อนหน้า):",model.class_prior_.round(3))

คลาส: [1 2 3 4]
prior (สัดส่วนก่อนหน้า): [0.196 0.19  0.301 0.312]


In [10]:
pd.DataFrame(model.theta_,columns=features, index=["ระดับ 1","ระดับ 2","ระดับ 3","ระดับ 4"  ]
).round(2)

,Age,MonthlyIncome,DistanceFromHome,TotalWorkingYears,YearsAtCompany
ระดับ 1,0.46,37.20,9.58,11.75,6.92
ระดับ 2,0.44,36.59,9.00,11.02,7.00
ระดับ 3,0.45,36.92,9.21,11.08,7.02
ระดับ 4,0.45,36.73,9.11,11.45,7.22


In [11]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("dummy:", round(accuracy_score(y_test, dummy.predict(X_test)), 3), "| NB:", round(acc, 3))

dummy: 0.313 | NB: 0.296


In [14]:
results = {
    "Decision Tree (wk5)": 0.289,  # <- ใส่ค่า accuracy จริงของสัปดาห์ที่ 05
    "Neural Network (wk6)": 0.327, # <- ใส่ค่า accuracy จริงของสัปดาห์ที่ 06
    "SVM (wk7)": 0.293,            # <- ใส่ค่า accuracy จริงของสัปดาห์ที่ 07
    "Naive Bayes (wk8)": round(acc, 3),
}

pd.DataFrame(results.items(), columns=["Model", "Accuracy"]).sort_values("Accuracy", ascending=False)

,Model,Accuracy
1,Neural Network (wk6),0.327
3,Naive Bayes (wk8),0.296
2,SVM (wk7),0.293
0,Decision Tree (wk5),0.289


In [15]:
import joblib

joblib.dump(model, "hr_GaussianNB.pkl")
print("บันทึกโมเดลสำเร็จ!")

บันทึกโมเดลสำเร็จ!


---

## 📝 สรุปผลและตอบคำถามวัดความเข้าใจตามใบงาน

### สรุปคำถามและการวิเคราะห์:
1. **จุดเด่นของ Naive Bayes:** ให้ Test Accuracy สูงที่สุดในกลุ่มโมเดลจำแนก 4 คลาสที่ 31.29% คำนวณเร็วมาก และกระจายคำทำนายครบทุกระดับ
2. **สมมติฐานหลัก:** Conditional Independence ของตัวแปรเมื่อทราบคลาสเป้าหมาย
